# Lab 2 · Fine-tune (LoRA / QLoRA) + MLflow — Kaggle T4

## What we're doing

We take a general chat model — **Qwen2.5-3B-Instruct** — and teach it **one
narrow behavior**: read a customer's free-text order message and reply with a
**strict JSON object**, nothing else. This is the classic *fine-tune for
behavior* case (vs. RAG, which is for *knowledge*). We're not giving the model
new facts; we're shaping *how* it answers.

## Why this is a good demo

Out of the box the base model tends to be chatty — it explains, wraps JSON in
```` ```json ```` fences, or invents extra keys. For a downstream program that
`json.loads()` the reply, that's broken. After a few minutes of LoRA training on
~1000 examples, it emits clean, schema-correct JSON. The before/after is obvious
and **measurable** (we put numbers on it in Lab 3).

## What you'll see

- **Cell 2** — the synthetic dataset and one real example printed out.
- **Cell 3 (BEFORE)** — base model output: often prose, fences, or wrong keys.
- **Cell 4 (TRAIN)** — QLoRA on a single T4 in a few minutes; prints peak VRAM.
- **Cell 5 (AFTER)** — same prompts, now tight JSON matching the schema.
- **Cell 6** — the run logged in MLflow (params + loss), reproducible.

---

**Run from the Kaggle editor on a T4** (Accelerator = *GPU T4 x2*, Internet =
*On*), then **Save Version → Save & Run All**. The API push can't request a T4
(it only sends `enable_gpu`, so it defaults to a P100 — which can't run Kaggle's
PyTorch). Editing happens in VS Code; the *run* is launched from the UI.

**Kaggle gotchas this notebook handles:** 'GPU T4 x2' = two T4s → we pin to one
(DataParallel breaks QLoRA); torch pinned to the preinstalled build; no bf16 on
T4 → fp16. Outputs (`adapters/lora`, `data/test.jsonl`, `mlflow.db`) land in
`/kaggle/working/` — **Lab 3 reads them as a kernel-source input.**

## Cell 1 — single GPU + pinned torch + sanity

In [ ]:
import os
# Kaggle 'GPU T4 x2' exposes TWO T4s; Trainer would shard via DataParallel, which
# breaks QLoRA 4-bit ('illegal memory access'). Pin to one GPU. (Set BEFORE torch.)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Pin torch to Kaggle's preinstalled build so pip can't swap in an incompatible wheel.
import torch
open("/tmp/constraints.txt", "w").write("torch==" + torch.__version__.split("+")[0] + "\n")
!pip -q install -c /tmp/constraints.txt trl peft bitsandbytes datasets accelerate mlflow

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("visible GPUs:", torch.cuda.device_count(), "| torch:", torch.__version__,
      "| capability sm_%d%d" % torch.cuda.get_device_capability())

## Cell 2 — the dataset: free-text order → strict JSON

We generate a **synthetic, seeded** dataset (same every run, so results are
reproducible). Each row is a 3-message chat:

- **system** — the instruction + the exact JSON schema to emit
- **user** — a natural order message, e.g. *"can I get 3 lavender shampoo to Hanoi"*
- **assistant** — the gold answer the model should learn:
  `{"intent":"order","item":"lavender shampoo","qty":3,"city":"Hanoi"}`

The schema has four fields: **intent** (`order`/`cancel`/`track`), **item**
(string), **qty** (integer), **city** (string or `null` if not mentioned). We
build ~1000 train + ~180 held-out test examples by combining 12 items, 6 cities,
and a handful of phrasings per intent, de-duplicated on the user text so the test
set isn't a leaked copy of training.

In [ ]:
import json, os, random
random.seed(0)  # deterministic — same dataset every run

ITEMS = ["lavender shampoo", "green tea", "running shoes", "USB-C cable",
         "yoga mat", "coffee beans", "phone case", "water bottle",
         "notebook", "wireless mouse", "desk lamp", "protein powder"]
CITIES = ["Hanoi", "Ho Chi Minh City", "Da Nang", "Singapore", "Tokyo", "Seattle"]
INTENTS = {
    "order":  ["I'd like to order {q} {item}", "can I get {q} {item}",
               "please send me {q} {item}", "order {q} {item} for me",
               "buy {q} {item}", "I want {q} {item} shipped to {city}"],
    "cancel": ["cancel my order of {item}", "I want to cancel the {item}",
               "please cancel {q} {item}", "stop the {item} order"],
    "track":  ["where is my {item}", "track my {item} order",
               "status of my {item}", "has my {item} shipped yet"],
}
SYSTEM = ('You are an order-intent parser. Reply with ONLY a JSON object with keys '
          '"intent" (order|cancel|track), "item" (string), "qty" (integer), '
          '"city" (string or null). No prose, no code fences.')

def make_example():
    intent = random.choice(list(INTENTS)); item = random.choice(ITEMS)
    qty = random.randint(1, 5); city = random.choice(CITIES)
    text = random.choice(INTENTS[intent]).format(q=qty, item=item, city=city)
    has_city = "{city}" in text or random.random() < 0.3
    if "{city}" not in text and has_city: text += f" to {city}"
    target = {"intent": intent, "item": item,
              "qty": qty if intent != "track" else 1,
              "city": city if has_city else None}
    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": text},
                         {"role": "assistant", "content": json.dumps(target, ensure_ascii=False)}]}

os.makedirs("data", exist_ok=True)
rows = [make_example() for _ in range(3000)]
seen, uniq = set(), []
for r in rows:
    k = r["messages"][1]["content"]
    if k not in seen: seen.add(k); uniq.append(r)
random.shuffle(uniq); split = int(len(uniq) * 0.85)
train, test = uniq[:split], uniq[split:]
for name, part in [("train", train), ("test", test)]:
    with open(f"data/{name}.jsonl", "w") as f:
        for r in part: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"data/{name}.jsonl: {len(part)} examples")

# Show one real example so you can see exactly what we're training on.
print("\n--- one training example (3-message chat) ---")
for m in train[0]["messages"]:
    print(f"[{m['role']:>9}] {m['content']}")

## Cell 3 — BEFORE: the base model on a few prompts

We feed the **system + user** turns (not the gold answer) to the *untrained* base
model and look at `OUT` vs `GOLD`. Watch for the failure modes we're trying to
fix: extra prose, ```` ```json ```` fences, missing/renamed keys, or the wrong
`city`/`qty`. This is the gap the fine-tune closes.

In [ ]:
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-3B-Instruct"   # one model across all labs
tests = [json.loads(l) for l in open("data/test.jsonl")]

def gen(tok, model, messages, max_new=128):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False)
    return tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

tok = AutoTokenizer.from_pretrained(MODEL)
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto").eval()  # fp16
print("== BASE model ==")
for t in tests[:5]:
    print("IN  :", t["messages"][1]["content"])
    print("OUT :", gen(tok, base, t["messages"][:2]))
    print("GOLD:", t["messages"][2]["content"], "\n")
del base; gc.collect(); torch.cuda.empty_cache()

## Cell 4 — Train (adaptive QLoRA/LoRA, fp16) tracked in MLflow

**LoRA** freezes the 3B base and trains tiny low-rank adapters (<1% of params) —
that's why this finishes in minutes on one GPU. **QLoRA** additionally loads the
frozen base in **4-bit** to save VRAM; we use it on T4-class GPUs (sm_75+) and
fall back to plain LoRA elsewhere. `report_to="mlflow"` logs params + the loss
curve. Watch the loss fall and note the **peak VRAM** printed at the end.

In [ ]:
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///" + os.path.abspath("mlflow.db")
os.environ["MLFLOW_EXPERIMENT_NAME"] = "qwen-order-parser"

from transformers import BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

OUT = "adapters/lora"
# 4-bit bitsandbytes is reliable on T4-class GPUs (sm_75+); P100 (sm_60) -> plain LoRA fp16.
use_qlora = torch.cuda.get_device_capability()[0] >= 7
method = "qlora" if use_qlora else "lora"
quant = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
         if use_qlora else None)
print(f"method: {method}  (base in {'4-bit' if use_qlora else 'fp16'})")

model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quant,
        dtype=torch.float16, device_map="auto")
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
ds = load_dataset("json", data_files={"train": "data/train.jsonl"})["train"]
# fp16=False on purpose: the trainable LoRA adapters are fp32 and the 4-bit base
# computes in fp16 via bnb, so the fp16 GradScaler isn't needed — and leaving it on
# triggers "_amp_foreach_non_finite_check_and_unscale_ not implemented for BFloat16".
cfg = SFTConfig(output_dir=OUT, num_train_epochs=2,            # trimmed for cloud speed
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, warmup_ratio=0.03, logging_steps=5,
        fp16=False, bf16=False, max_length=512,                # no GradScaler (see note above)
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        report_to="mlflow", run_name=f"{method}-r16", save_strategy="epoch")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora)
print(f"== training {method} on {len(ds)} examples ==")
trainer.train()
trainer.save_model(OUT); tok.save_pretrained(OUT)
print(f"adapter saved to {OUT} | peak VRAM {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
del trainer, model; gc.collect(); torch.cuda.empty_cache()

## Cell 5 — AFTER: base vs fine-tuned, side by side

Same prompts as the BEFORE cell, now through the fine-tuned adapter. You should
see `OUT` collapse to clean JSON that matches `GOLD` — no prose, no fences. (Lab
3 turns this eyeball check into hard numbers on the full held-out set.)

In [ ]:
from peft import PeftModel
ft = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16, device_map="auto"),
        OUT).eval()
print("== FINE-TUNED ==")
for t in tests[:5]:
    print("IN  :", t["messages"][1]["content"])
    print("OUT :", gen(tok, ft, t["messages"][:2]))
    print("GOLD:", t["messages"][2]["content"], "\n")

## Cell 6 — MLflow: what got logged

Kaggle is batch, so there's no live MLflow UI — but the run is fully logged to
`mlflow.db`. Download it from the kernel output and run `mlflow ui
--backend-store-uri sqlite:///mlflow.db` locally to see the loss curve.

In [ ]:
import mlflow
mlflow.set_tracking_uri("sqlite:///" + os.path.abspath("mlflow.db"))
exp = mlflow.get_experiment_by_name("qwen-order-parser")
runs = mlflow.search_runs([exp.experiment_id])
cols = [c for c in runs.columns if c.startswith(("metrics.", "params.")) ][:12]
print(runs[["run_id"] + cols].T)
print("\nadapter + data + mlflow.db are in /kaggle/working — Lab 3 reads them as a kernel source.")